# SENSEI — Session Intelligence
## Module 1: Purchase Prediction · Classification

> **Module 1** is the first SENSEI intelligence module.  
> It answers the most business-critical session question:  
> **"Is this visitor about to buy?"**

Using the SENSEI session feature store (built in notebook 02), we train and evaluate
binary classifiers to predict whether a session ends in a purchase.

**Challenge:** Only 0.81 % of sessions result in a purchase — a 1:122 class imbalance.  
We handle this with `class_weight='balanced'` and evaluate with PR-AUC and F1, not accuracy.

## Contents
1. Load SENSEI feature store & train/test split
2. Baseline — Logistic Regression
3. Random Forest
4. Evaluation: normalized confusion matrix + PR curve
5. Cross-validation (5-fold stratified)
6. Permutation feature importance
7. Threshold tuning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    f1_score,
    ConfusionMatrixDisplay,
)

sns.set_theme(style='whitegrid', palette='muted')
DATA_DIR = os.path.join('..', 'data')
RANDOM_STATE = 42

## 1. Load Data & Train/Test Split

In [ ]:
sessions = pd.read_parquet(os.path.join(DATA_DIR, 'sessions_features.parquet'))
print(f'Shape: {sessions.shape}')
sessions.head()

In [ ]:
FEATURES = [
    'n_views',
    'n_addtocart',
    'n_items',
    'n_revisited_items',
    'duration_sec',
    'hour_of_day',
    'day_of_week',
    'view_to_cart_ratio',
    'is_first_session',
]
TARGET = 'purchased'

X = sessions[FEATURES]
y = sessions[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows')
print(f'Purchase rate — train: {y_train.mean()*100:.2f}%  |  test: {y_test.mean()*100:.2f}%')

In [ ]:
# Scale for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## 2. Baseline: Logistic Regression

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_scaled, y_train)

y_pred_lr   = lr.predict(X_test_scaled)
y_proba_lr  = lr.predict_proba(X_test_scaled)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['no purchase', 'purchase']))

## 3. Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    max_depth=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['no purchase', 'purchase']))

## 4. Evaluation & Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Normalized confusion matrices ---
for ax, y_pred, label in [
    (axes[0], y_pred_lr, 'Logistic Regression'),
    (axes[1], y_pred_rf, 'Random Forest'),
]:
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=['No purchase', 'Purchase'],
        normalize='true',          # show recall per class, not raw counts
        ax=ax, colorbar=False, cmap='Blues',
        values_format='.2f',
    )
    ax.set_title(f'{label}\n(normalized by true label)')

# --- Precision-Recall curves ---
for y_prob, label, color in [
    (y_proba_lr, 'Logistic Regression', '#4C72B0'),
    (y_proba_rf, 'Random Forest',        '#55A868'),
]:
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    axes[2].plot(rec, prec, label=f'{label} (AP={ap:.3f})', color=color)

baseline_rate = y_test.mean()
axes[2].axhline(baseline_rate, linestyle='--', color='grey', label=f'No-skill baseline ({baseline_rate:.3f})')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
results = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'ROC-AUC': roc_auc_score(y_test, y_proba_lr),
        'PR-AUC':  average_precision_score(y_test, y_proba_lr),
        'F1 (purchase)': f1_score(y_test, y_pred_lr),
    },
    {
        'Model': 'Random Forest',
        'ROC-AUC': roc_auc_score(y_test, y_proba_rf),
        'PR-AUC':  average_precision_score(y_test, y_proba_rf),
        'F1 (purchase)': f1_score(y_test, y_pred_rf),
    },
]).set_index('Model')

print(results.round(4))

## 4b. Cross-Validation (5-Fold Stratified)

In [ ]:
# Permutation importance is more reliable than MDI (Mean Decrease in Impurity)
# because MDI is biased towards high-cardinality numerical features.
perm = permutation_importance(
    rf, X_test, y_test,
    n_repeats=10,
    scoring='average_precision',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

perm_df = (
    pd.DataFrame({'feature': FEATURES, 'importance': perm.importances_mean})
    .sort_values('importance')
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=perm_df, x='importance', y='feature', ax=ax, palette='muted')
ax.set_title('Permutation Importance — Random Forest\n(scored by PR-AUC drop)')
ax.set_xlabel('Mean PR-AUC decrease')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## Summary & Interpretation

| Model | ROC-AUC | PR-AUC | F1 @ 0.5 threshold |
|-------|---------|--------|--------------------|
| Logistic Regression | — | — | — |
| Random Forest | — | — | — |

*(values fill in after running)*

### Design decisions & why they matter

| Decision | Reason |
|---|---|
| Removed `n_events` | Linear combination of `n_views + n_addtocart` — pure multicollinearity |
| Removed `has_addtocart` | 100 % redundant with `n_addtocart` |
| `class_weight='balanced'` | Compensates for 1:122 imbalance without data augmentation |
| Normalized confusion matrix | Raw counts are misleading with 99 % majority class |
| Permutation importance | MDI over-weights high-cardinality features; permutation is model-agnostic |
| Threshold tuning | Default 0.5 is almost always wrong for imbalanced targets |
| 5-fold stratified CV | Single split is not robust; stratification preserves class ratio per fold |

### Possible next steps
- Gradient boosting (LightGBM / XGBoost) — typically outperforms RF on tabular data
- Add item-category features from `item_properties` files
- SMOTE as an alternative to `class_weight` (compare PR-AUC)
- Use the tuned threshold from the validation set, not the test set, in production

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, y_prob, label, color in [
    (axes[0], y_proba_lr, 'Logistic Regression', '#4C72B0'),
    (axes[1], y_proba_rf, 'Random Forest',        '#55A868'),
]:
    prec, rec, thresholds = precision_recall_curve(y_test, y_prob)
    # F1 at each threshold (avoid division by zero)
    f1_scores = np.where(
        prec[:-1] + rec[:-1] > 0,
        2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1]),
        0
    )
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]

    threshold_df = pd.DataFrame({
        'Threshold': thresholds,
        'Precision': prec[:-1],
        'Recall':    rec[:-1],
        'F1':        f1_scores,
    })

    ax.plot(thresholds, prec[:-1],  label='Precision', color='#4C72B0')
    ax.plot(thresholds, rec[:-1],   label='Recall',    color='#DD8452')
    ax.plot(thresholds, f1_scores,  label='F1',        color='#55A868', linewidth=2)
    ax.axvline(best_threshold, linestyle='--', color='grey',
               label=f'Best threshold={best_threshold:.2f} (F1={best_f1:.3f})')
    ax.set_title(f'{label} — Threshold tuning')
    ax.set_xlabel('Classification threshold')
    ax.set_ylabel('Score')
    ax.legend(fontsize=8)

    print(f'{label}: best threshold = {best_threshold:.3f}  →  F1 = {best_f1:.4f}')
    y_pred_tuned = (y_prob >= best_threshold).astype(int)
    print(classification_report(y_test, y_pred_tuned, target_names=['no purchase', 'purchase']))

plt.tight_layout()
plt.show()

## 5. Feature Importance (Random Forest)

In [ ]:
importances = (
    pd.Series(rf.feature_importances_, index=FEATURES)
    .sort_values()
    .reset_index()
)
importances.columns = ['feature', 'importance']

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=importances, x='importance', y='feature', ax=ax, palette='muted')
ax.set_title('Feature Importance — Random Forest')
ax.set_xlabel('Mean decrease in impurity')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## Summary & Interpretation

| Model | ROC-AUC | PR-AUC | F1 (purchase) |
|-------|---------|--------|---------------|
| Logistic Regression | — | — | — |
| Random Forest | — | — | — |

*(values will fill in after running)*

### Key takeaways
- **Accuracy is misleading** here — a model predicting "no purchase" for every session would achieve 99.19 % accuracy.
- **PR-AUC** and **F1** are the right metrics for imbalanced problems.
- `has_addtocart` and `n_addtocart` are expected to be the strongest predictors — adding items to cart is a clear purchase signal.
- `duration_sec` captures engagement depth that raw event counts miss.

### Possible next steps
- Add time-of-day / day-of-week features from session start time
- Add item-category features (from `item_properties` files)
- Try gradient boosting (XGBoost / LightGBM) for better performance
- Tune classification threshold to optimise Precision vs. Recall trade-off